# 02 -- Generative Adversarial Networks

**Goal:** train a generator $G$ that maps noise to images by pitting it against a discriminator $D$
that tries to tell real images from generated ones.

**In this notebook:** the minimax game and why the non-saturating loss is used, DCGAN design rules,
WGAN-GP, DiffAugment for small datasets like Ray_dog, mode collapse, and Hub upload.

*Papers:* Goodfellow et al. 2014 · Radford et al. (DCGAN) 2015 · Gulrajani et al. (WGAN-GP) 2017 · Zhao et al. (DiffAugment) 2020.

## 1. Theory

### The game
$$
\min_G \max_D \; V(D,G)=\mathbb E_{x\sim p_\text{data}}[\log D(x)] + \mathbb E_{z\sim\mathcal N(0,I)}[\log(1-D(G(z)))]
$$
For a fixed $G$ the optimal discriminator is $D^*(x)=\frac{p_\text{data}(x)}{p_\text{data}(x)+p_G(x)}$, and plugging it
in shows that $G$ minimises the **Jensen-Shannon divergence** between $p_\text{data}$ and $p_G$.
There is no likelihood and no reconstruction: $G$ never sees a real image, only $D$'s gradients.

### Non-saturating loss
Early on, $D$ easily rejects fakes, $D(G(z))\approx 0$, and $\log(1-D(G(z)))$ is flat -- $G$ gets no gradient.
In practice $G$ instead **maximises $\log D(G(z))$** (same fixed point, strong gradients when $G$ is bad).
In logits: $\mathcal L_D=\operatorname{softplus}(-D(x))+\operatorname{softplus}(D(G(z)))$, $\mathcal L_G=\operatorname{softplus}(-D(G(z)))$.

### DCGAN rules that make it train
Strided (transposed) convolutions instead of pooling, BatchNorm in both nets (not on $D$'s input
or $G$'s output), ReLU in $G$ with a `tanh` output, LeakyReLU in $D$, Adam with lr $2\cdot10^{-4}$ and $\beta_1=0.5$,
weights initialised from $\mathcal N(0,0.02)$.

### WGAN-GP
Replace JS with the **Wasserstein-1 distance**, $W(p,q)=\sup_{\|f\|_L\le1}\mathbb E_p f-\mathbb E_q f$. $D$ becomes a
*critic* (no sigmoid), kept 1-Lipschitz with a **gradient penalty** $\lambda\,\mathbb E_{\hat x}(\|\nabla_{\hat x}D(\hat x)\|-1)^2$
on random interpolates of real and fake. The critic's estimate (`wasserstein` in the logs) is a loss
that actually tracks sample quality. Set `method.loss: wgan-gp`.

### Failure modes to watch for
* **Mode collapse:** $G$ produces a few images that fool $D$; diversity disappears from the grid.
* **D wins:** `d_real -> 1`, `d_fake -> 0` and `g_loss` keeps climbing. With small data this is $D$ **memorising**
  the training set -- the reason for DiffAugment (below).

### DiffAugment (small datasets)
Apply the same random, *differentiable* augmentation (colour, translation, cutout) to real **and** fake images
every time $D$ sees them. $D$ can no longer memorise 258 photos, and because the augmentation is applied to
both sides $G$ is not pushed to generate augmented-looking images. Enabled in `configs/ray_dog/dcgan.yaml`.

## 0. Setup

Works in three places, no edits needed:

* **Google Colab** -- the cell below downloads the code from GitHub and installs it (~1 min).
  Choose a GPU first: *Runtime -> Change runtime type -> T4 GPU*.
* **RunPod** -- run `bash scripts/runpod/setup.sh` once, then open JupyterLab (port 8888).
* **Your own machine** -- `pip install -e ".[notebook,finetune]"` in the repo, then open the notebook.

In [ ]:
import sys, pathlib, subprocess

# Where is the code? If this notebook was opened on its own (e.g. in Colab), fetch it first.
REPO_URL = "https://github.com/yar78/vision-generative-models-survey.git"
BRANCH = "claude/vibrant-noether-ftcsah"   # change to "main" once this branch is merged

REPO = pathlib.Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
if not (REPO / "genmodels").is_dir():
    REPO = pathlib.Path("vision-generative-models-survey").resolve()
    if not REPO.is_dir():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}"], check=True)
sys.path.insert(0, str(REPO))

import torch
import yaml
from genmodels import build_trainer
from genmodels.common.config import load_config
from genmodels.common.data import get_dataset
from genmodels.common.utils import get_device, load_images, nearest_neighbours, plot_history, show_grid
from genmodels.hub import push_run, resolve_repo_id
from genmodels.gan import Generator, d_loss_bce, g_loss_bce, gradient_penalty
from genmodels.common.diffaugment import diff_augment
print("code:", REPO)
print("device:", get_device(), f"({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "-- no GPU: training will be slow")

### The losses, as implemented

In [ ]:
import inspect
for f in (d_loss_bce, g_loss_bce, gradient_penalty): print(inspect.getsource(f))

## 2. Choose an experiment

Every experiment is one YAML file in `configs/`, shared with the command-line script,
so a run started here and a run started on a RunPod GPU are the same experiment.

* `configs/gan/dcgan_mnist.yaml` -- the quick default.
* `configs/ray_dog/dcgan.yaml` -- your **Ray_dog** photos (`yaryar78/Ray_dog`, 258 images) at 64x64.
* `configs/gan/wgangp_cifar10.yaml` -- WGAN-GP on CIFAR-10.

Edit `cfg` below to change anything (steps, batch size, model width ...).

In [ ]:
CONFIG = REPO / "configs/gan/dcgan_mnist.yaml"        # or: REPO / "configs/ray_dog/dcgan.yaml"
cfg = load_config(CONFIG)

# Override anything here, for example:
# cfg["train"]["steps"] = 5000

print(yaml.safe_dump(cfg, sort_keys=False))

### Look at the training data (with the training-time augmentation applied)

In [ ]:
train_set = get_dataset(dict(cfg["data"], num_workers=0))
print(f"{len(train_set)} training images")
x_real = torch.stack([train_set[i][0] for i in torch.randperm(len(train_set))[:64].tolist()])
show_grid(x_real, title="training data")

### What DiffAugment does to a batch
This is what the discriminator sees when `method.diffaugment` is set (the generator's output gets the same treatment).

In [ ]:
show_grid(diff_augment(x_real[:16], 'color,translation,cutout'), nrow=8, title='DiffAugment: color, translation, cutout')

## 3. Build the model

`build_trainer(cfg)` creates the generator and discriminator, the optimizers and the training loop.
Outputs go to `outputs/<name>/` (on RunPod: `/workspace/genmodels/outputs/<name>/`, on the network volume).

In [ ]:
trainer = build_trainer(cfg)
print("run dir:", trainer.run_dir)
print({k: f"{v/1e6:.2f}M params" for k, v in trainer.num_params().items()})

In [ ]:
print(trainer.G)
print(trainer.D)

## 4. Train

`trainer.fit()` runs `train.steps` steps. Every `sample_every` steps it saves a fixed-seed
sample grid to `samples/`; every `ckpt_every` steps a checkpoint. Re-running the cell
(or the script) **resumes** from the checkpoint.

MNIST DCGAN: ~10 min on a GPU for 20k steps. Healthy BCE training: `d_real` around 0.6-0.8, `d_fake` around 0.2-0.4, both losses noisy but not drifting.

Already trained this config with the script (e.g. on RunPod)? Set `TRAIN = False`
and the checkpoint in the run directory is loaded instead.

### Start TensorBoard first, then train

Run this cell **before** the training cell. The dashboard updates live while training runs
(it refreshes every 30 s; the reload button at its top right forces it):

* **Scalars** -- the loss curves.
* **Images** -- the fixed-seed sample grid, with a slider over training steps.

On RunPod with the script, TensorBoard is also served on port 6006 by `scripts/runpod/train.sh`.

Started training without it? Stop the training cell (■), run this one, then run the training
cell again: it **continues from the same step** -- the model and optimizer are still in memory.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {trainer.run_dir}/tensorboard

In [ ]:
TRAIN = True

if TRAIN:
    trainer.fit()
elif trainer.ckpt_path.exists():
    trainer.load_checkpoint()
    print(f"loaded checkpoint at step {trainer.step}")
else:
    print("No checkpoint yet in", trainer.run_dir)

### Loss curves
The same curves as TensorBoard's Scalars tab, drawn here once training has stopped.

In [ ]:
plot_history(trainer.history)

## 5. Results
### Samples (fixed noise)

In [ ]:
show_grid(trainer.sample_fixed(64), title='samples')

### How the samples evolved
The same 64 noise vectors at each saved step. Sudden loss of variety = mode collapse.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
grids = sorted((trainer.run_dir / "samples").glob("step_*.png"))
pick = grids[:: max(1, len(grids) // 4)][-4:] if grids else []
fig, axes = plt.subplots(1, len(pick), figsize=(4 * len(pick), 4), squeeze=False)
for ax, p in zip(axes[0], pick):
    ax.imshow(Image.open(p)); ax.set_title(p.stem); ax.axis("off")
plt.show()

## 6. Latent space
Interpolating between two noise vectors. Smooth morphs mean $G$ has learnt a continuous mapping rather than memorising discrete images.

In [ ]:
G = trainer.G.eval()
with torch.no_grad():
    za, zb = torch.randn(2, 6, G.z_dim, device=trainer.device)
    alphas = torch.linspace(0, 1, 10, device=trainer.device)
    z = torch.cat([(1 - a) * za + a * zb for a in alphas]).view(10, 6, -1).transpose(0, 1).reshape(60, -1)
    y = (torch.arange(60, device=trainer.device) // 10) % max(1, G.num_classes) if G.num_classes else None
    show_grid(G(z, y), nrow=10, title="z interpolations", size=10)

### Memorisation check

With a small dataset (Ray_dog has 258 images) a generative model can simply copy training
images. For each sample (top row) we show its nearest training image in pixel space (bottom
row). Near-identical pairs mean memorisation; similar-but-different means the model generalises.

In [ ]:
eval_set = get_dataset(dict(cfg["data"], random_crop=False, hflip=False, num_workers=0))
train_imgs = load_images(eval_set, max_images=10_000)
samples = trainer.sample_fixed(8).cpu()
nn_imgs = nearest_neighbours(samples, train_imgs)[:, 0]
show_grid(torch.cat([samples, nn_imgs]), nrow=8, title="samples (top) vs nearest training image (bottom)", size=10)

## 7. Push to the Hugging Face Hub

`push_run` uploads the run directory as a model repo **`<your-username>/{cfg['name']}`**
(private by default):

| file | |
|---|---|
| `model.safetensors`, `config.json` | weights + constructor args (`Generator.from_pretrained` reads these) |
| `README.md` | generated model card: dataset, steps, hardware, git commit, metrics |
| `samples.png`, `metrics.json`, `train_config.yaml` | results and the exact config |
| `runs/` | TensorBoard logs (shown in the Hub's *Training metrics* tab) |

Login once with a **write** token from https://huggingface.co/settings/tokens
(`notebook_login()` below, `huggingface-cli login`, or the `HF_TOKEN` env var). To use another
repo name set `cfg["hub"]["repo_id"]`; to publish, `cfg["hub"]["private"] = False` before training,
or `push_run(..., private=False)`.

In [ ]:
PUSH_TO_HUB = False

if PUSH_TO_HUB:
    from huggingface_hub import notebook_login, whoami
    try:
        print("logged in as", whoami()["name"])
    except Exception:
        notebook_login()
    url = push_run(trainer.run_dir)

### Reload it from the Hub
This is how anyone (including future you) uses the model.

In [ ]:
if PUSH_TO_HUB:
    from genmodels.gan import Generator
    model = Generator.from_pretrained(resolve_repo_id(cfg)).to(trainer.device).eval()
    show_grid(model.generate(16), title="reloaded from the Hub")

## 8. Things to try

* Ray_dog **without** DiffAugment (`cfg["method"]["diffaugment"] = None`): watch `d_real -> 1` and the samples collapse.
* `method.conditional: true` on MNIST (a cGAN), then `model.generate(10, y=torch.arange(10))`.
* `configs/gan/wgangp_cifar10.yaml`: does the `wasserstein` estimate fall as samples improve?